# Sistema predictivo de recomendación de cultivos para la optimización agrícola
## Segunda Entrega: Modelado y Evaluación de Clasificadores Base

**Integrantes:**
- Jonatan Romero (`jonatan.romeroa@udea.edu.co`)
- Sebastian Berrio (`sebastian.berriom@udea.edu.co`)
- Cristian Diez (`cristian.diez@udea.edu.co`)

*Ingeniería de Sistemas — Universidad de Antioquia*

---

### 1. Configuración del entorno y librerías

Se importan las librerías necesarias para el entrenamiento, validación cruzada y evaluación de los cinco clasificadores definidos en el diseño experimental: Regresión Logística, KNN, Random Forest, Perceptrón Multicapa (MLP) y SVM.

Se fija la semilla global `random_state=42` para garantizar la reproducibilidad en la partición de los datos y en el ajuste de los modelos.

In [1]:
from inspect import signature
from pathlib import Path
from typing import Any
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.precision", 4)

### 2. Carga del conjunto de datos y definición de variables

Se carga el archivo `data/Crop_recommendation.csv`, compuesto por 2200 registros distribuidos de forma balanceada en 22 cultivos (100 muestras por clase, sin valores faltantes ni registros duplicados).

Se depuran las columnas vacías residuales del archivo fuente y se separan las 7 variables predictoras numéricas (`N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`) en la matriz `X` y el cultivo objetivo en el vector `y` (`label`).

In [2]:
ruta_dataset = Path("data/Crop_recommendation.csv")
if not ruta_dataset.exists():
    ruta_dataset = Path("../data/Crop_recommendation.csv")

df = pd.read_csv(ruta_dataset)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df = df.rename(columns={"Nitrogen": "N", "phosphorus": "P", "potassium": "K"})

columnas_predictoras: list[str] = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall",
]

X: pd.DataFrame = df[columnas_predictoras].copy()
y: pd.Series = df["label"].copy()

resumen_datos = pd.DataFrame(
    {
        "Métrica": [
            "Total de muestras",
            "Variables predictoras (X)",
            "Número de clases (y)",
            "Muestras por clase",
        ],
        "Valor": [
            X.shape[0],
            X.shape[1],
            y.nunique(),
            int(y.value_counts().iloc[0]),
        ],
    }
)
display(resumen_datos)
display(X.head())

,Métrica,Valor
0,Total de muestras,2200
1,Variables predictoras (X),7
2,Número de clases (y),22
3,Muestras por clase,100


,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,20.8797,82.0027,6.5030,202.9355
1,85,58,41,21.7705,80.3196,7.0381,226.6555
2,60,55,44,23.0045,82.3208,7.8402,263.9642
3,74,35,40,26.4911,80.1584,6.9804,242.8640
4,78,42,42,20.1302,81.6049,7.6285,262.7173


### 3. Partición estratificada 80/20 y validación cruzada

El conjunto de datos se divide en 80% para entrenamiento (1760 muestras, 80 por cultivo) y 20% para prueba (440 muestras, 20 por cultivo) mediante muestreo estratificado (`stratify=y`), conservando el balance de las 22 clases en ambos subconjuntos.

Sobre los datos de entrenamiento se configura `StratifiedKFold` de 5 particiones para la búsqueda en malla (`GridSearchCV`) y se define una función auxiliar que evalúa cada modelo optimizado sobre el conjunto de prueba con las métricas requeridas: Accuracy, Error de Clasificación ($1 - \text{Accuracy}$), Macro Precision, Macro Recall y Macro F1.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

cv_estratificada = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

resultados_evaluacion: list[dict[str, Any]] = []
modelos_optimizados: dict[str, GridSearchCV] = {}


def entrenar_y_evaluar_modelo(
    nombre_modelo: str,
    estimador: Any,
    rejilla_parametros: dict[str, list[Any]],
) -> pd.DataFrame:
    busqueda = GridSearchCV(
        estimator=estimador,
        param_grid=rejilla_parametros,
        cv=cv_estratificada,
        scoring="f1_macro",
        n_jobs=-1,
        refit=True,
    )
    busqueda.fit(X_train, y_train)
    y_pred = busqueda.predict(X_test)

    acc = float(accuracy_score(y_test, y_pred))
    error_clasificacion = 1.0 - acc
    macro_prec = float(precision_score(y_test, y_pred, average="macro", zero_division=0))
    macro_rec = float(recall_score(y_test, y_pred, average="macro", zero_division=0))
    macro_f1 = float(f1_score(y_test, y_pred, average="macro", zero_division=0))

    parametros_limpios = {
        k.split("__")[-1]: v for k, v in busqueda.best_params_.items()
    }

    registro = {
        "Modelo": nombre_modelo,
        "Mejor Hiperparámetro": str(parametros_limpios),
        "Accuracy": acc,
        "Error de Clasificación (1 - Accuracy)": error_clasificacion,
        "Macro Precision": macro_prec,
        "Macro Recall": macro_rec,
        "Macro F1": macro_f1,
    }

    resultados_evaluacion.append(registro)
    modelos_optimizados[nombre_modelo] = busqueda
    return pd.DataFrame([registro])


particion_df = pd.DataFrame(
    {
        "Subconjunto": ["Entrenamiento (80%)", "Prueba (20%)"],
        "Muestras": [X_train.shape[0], X_test.shape[0]],
        "Clases representadas": [y_train.nunique(), y_test.nunique()],
        "Muestras por clase": [
            int(y_train.value_counts().iloc[0]),
            int(y_test.value_counts().iloc[0]),
        ],
    }
)
display(particion_df)

,Subconjunto,Muestras,Clases representadas,Muestras por clase
0,Entrenamiento (80%),1760,22,80
1,Prueba (20%),440,22,20


### 4. Regresión Logística (Modelo Paramétrico)

La Regresión Logística multinomial estima fronteras de decisión lineales entre las 22 clases optimizando la pérdida de entropía cruzada con el algoritmo `lbfgs`.

Debido a las escalas heterogéneas observadas en el análisis exploratorio (por ejemplo, `ph` entre 3.5 y 9.9 frente a `rainfall` cercano a 300 mm), se incorpora `StandardScaler` dentro de un `Pipeline` para ajustar la estandarización únicamente sobre los pliegues de entrenamiento y evitar fuga de información mientras se evalúa `C` en `[0.01, 0.1, 1, 10, 100]`.

In [4]:
parametros_lr: dict[str, Any] = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": RANDOM_STATE,
}
if "multi_class" in signature(LogisticRegression).parameters:
    parametros_lr["multi_class"] = "multinomial"

pipeline_logistica = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(**parametros_lr)),
    ]
)

rejilla_logistica: dict[str, list[Any]] = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
}

resultado_logistica = entrenar_y_evaluar_modelo(
    nombre_modelo="Regresión Logística",
    estimador=pipeline_logistica,
    rejilla_parametros=rejilla_logistica,
)
display(resultado_logistica)

,Modelo,Mejor Hiperparámetro,Accuracy,Error de Clasificación (1 - Accuracy),Macro Precision,Macro Recall,Macro F1
0,Regresión Logística,{'C': 100},0.9841,0.0159,0.9852,0.9841,0.984


### 5. K-Nearest Neighbors - KNN (Modelo No Paramétrico)

KNN clasifica cada observación según la clase mayoritaria entre sus `n_neighbors` vecinos más cercanos usando distancia euclidiana, sin asumir una forma paramétrica previa para las clases.

Como la métrica de distancia es sensible a la magnitud de cada variable, se integra `StandardScaler` en el `Pipeline` y se evalúan mediante `GridSearchCV` los valores de `n_neighbors` en `[4, 13, 14, 15, 16, 17]`.

In [5]:
pipeline_knn = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier()),
    ]
)

rejilla_knn: dict[str, list[Any]] = {
    "clf__n_neighbors": [4, 13, 14, 15, 16, 17],
}

resultado_knn = entrenar_y_evaluar_modelo(
    nombre_modelo="K-Nearest Neighbors (KNN)",
    estimador=pipeline_knn,
    rejilla_parametros=rejilla_knn,
)
display(resultado_knn)

,Modelo,Mejor Hiperparámetro,Accuracy,Error de Clasificación (1 - Accuracy),Macro Precision,Macro Recall,Macro F1
0,K-Nearest Neighbors (KNN),{'n_neighbors': 4},0.9727,0.0273,0.9746,0.9727,0.9725


### 6. Random Forest (Modelo de Ensamble)

Random Forest combina múltiples árboles de decisión entrenados sobre muestras *bootstrap* y subconjuntos aleatorios de variables, lo que reduce la varianza y permite capturar relaciones no lineales entre nutrientes (NPK) y condiciones climáticas.

Al dividir el espacio mediante umbrales por variable, no requiere estandarización previa ni se ve afectado por los valores atípicos informativos detectados en `K` y `P`. Se optimizan `n_estimators` en `[18, 19]`, `max_depth` en `[5, 10, 20, None]` y `max_features` en `['sqrt', 'log2']`.

In [6]:
estimador_rf = RandomForestClassifier(random_state=RANDOM_STATE)

rejilla_rf: dict[str, list[Any]] = {
    "n_estimators": [18, 19],
    "max_depth": [5, 10, 20, None],
    "max_features": ["sqrt", "log2"],
}

resultado_rf = entrenar_y_evaluar_modelo(
    nombre_modelo="Random Forest",
    estimador=estimador_rf,
    rejilla_parametros=rejilla_rf,
)
display(resultado_rf)

,Modelo,Mejor Hiperparámetro,Accuracy,Error de Clasificación (1 - Accuracy),Macro Precision,Macro Recall,Macro F1
0,Random Forest,"{'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 19}",0.9886,0.0114,0.9897,0.9886,0.9887


### 7. Perceptrón Multicapa - MLP (Red Neuronal Artificial)

El Perceptrón Multicapa aprende fronteras de decisión no lineales mediante capas ocultas entrenadas por retropropagación con un máximo de 500 iteraciones (`max_iter=500`).

Se incluye `StandardScaler` en el `Pipeline` para estabilizar la convergencia del optimizador y se exploran las configuraciones de capas ocultas `hidden_layer_sizes` en `[(16,), (32,), (16, 16), (32, 32), (32, 16)]` junto con la regularización `alpha` en `[0.0001, 0.001, 0.01]`.

In [7]:
pipeline_mlp = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            MLPClassifier(
                max_iter=500,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

rejilla_mlp: dict[str, list[Any]] = {
    "clf__hidden_layer_sizes": [(16,), (32,), (16, 16), (32, 32), (32, 16)],
    "clf__alpha": [0.0001, 0.001, 0.01],
}

resultado_mlp = entrenar_y_evaluar_modelo(
    nombre_modelo="Perceptrón Multicapa (MLP)",
    estimador=pipeline_mlp,
    rejilla_parametros=rejilla_mlp,
)
display(resultado_mlp)

,Modelo,Mejor Hiperparámetro,Accuracy,Error de Clasificación (1 - Accuracy),Macro Precision,Macro Recall,Macro F1
0,Perceptrón Multicapa (MLP),"{'alpha': 0.01, 'hidden_layer_sizes': (32,)}",0.9932,0.0068,0.9937,0.9932,0.9932


### 8. Support Vector Machine - SVM con Kernel RBF

La máquina de soporte vectorial con *kernel* de base radial (RBF) proyecta las siete variables hacia un espacio de mayor dimensión para separar cultivos cuyas condiciones agroclimáticas no son linealmente separables.

Dada la dependencia del *kernel* RBF respecto a las distancias entre muestras, se aplica `StandardScaler` dentro del `Pipeline` y se ajusta la rejilla compuesta por `C` en `[0.1, 1, 10, 100]` y `gamma` en `['scale', 0.01, 0.1, 1]`.

In [8]:
pipeline_svm = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            SVC(
                kernel="rbf",
                probability=True,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

rejilla_svm: dict[str, list[Any]] = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.01, 0.1, 1],
}

resultado_svm = entrenar_y_evaluar_modelo(
    nombre_modelo="Support Vector Machine (SVM RBF)",
    estimador=pipeline_svm,
    rejilla_parametros=rejilla_svm,
)
display(resultado_svm)

,Modelo,Mejor Hiperparámetro,Accuracy,Error de Clasificación (1 - Accuracy),Macro Precision,Macro Recall,Macro F1
0,Support Vector Machine (SVM RBF),"{'C': 1, 'gamma': 1}",0.9909,0.0091,0.9915,0.9909,0.9909


### 9. Tabla resumen de resultados en el conjunto de prueba

Se consolidan en un `DataFrame` las métricas obtenidas por los cinco modelos optimizados al evaluar el 20% del conjunto de prueba reservado (440 muestras).

La tabla se ordena de forma descendente según `Macro F1` y `Accuracy` para identificar directamente los dos modelos con mejor desempeño.

In [9]:
columnas_resumen: list[str] = [
    "Modelo",
    "Mejor Hiperparámetro",
    "Accuracy",
    "Error de Clasificación (1 - Accuracy)",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
]

df_resultados_consolidados = (
    pd.DataFrame(resultados_evaluacion)[columnas_resumen]
    .sort_values(by=["Macro F1", "Accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(df_resultados_consolidados)

mejores_dos_modelos = df_resultados_consolidados.head(2)[
    ["Modelo", "Mejor Hiperparámetro", "Accuracy", "Macro F1"]
]
display(mejores_dos_modelos)

,Modelo,Mejor Hiperparámetro,Accuracy,Error de Clasificación (1 - Accuracy),Macro Precision,Macro Recall,Macro F1
0,Perceptrón Multicapa (MLP),"{'alpha': 0.01, 'hidden_layer_sizes': (32,)}",0.9932,0.0068,0.9937,0.9932,0.9932
1,Support Vector Machine (SVM RBF),"{'C': 1, 'gamma': 1}",0.9909,0.0091,0.9915,0.9909,0.9909
2,Random Forest,"{'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 19}",0.9886,0.0114,0.9897,0.9886,0.9887
3,Regresión Logística,{'C': 100},0.9841,0.0159,0.9852,0.9841,0.9840
4,K-Nearest Neighbors (KNN),{'n_neighbors': 4},0.9727,0.0273,0.9746,0.9727,0.9725


,Modelo,Mejor Hiperparámetro,Accuracy,Macro F1
0,Perceptrón Multicapa (MLP),"{'alpha': 0.01, 'hidden_layer_sizes': (32,)}",0.9932,0.9932
1,Support Vector Machine (SVM RBF),"{'C': 1, 'gamma': 1}",0.9909,0.9909


### 10. Conclusión: Selección de los dos mejores modelos base

Los resultados sobre el conjunto de prueba muestran que los cinco clasificadores superan el 97% de exactitud, lo cual concuerda con la alta separabilidad observada en el análisis exploratorio para variables como potasio (`K`), humedad (`humidity`) y fósforo (`P`). De acuerdo con el **F1-Score Macro** y el **Accuracy**, los **dos mejores modelos** son:

1. **Perceptrón Multicapa (MLP):** Ocupa el primer lugar con los hiperparámetros `{'alpha': 0.01, 'hidden_layer_sizes': (32,)}`, alcanzando un **Accuracy de 0.9932** (Error de clasificación de **0.0068**) y un **Macro F1 de 0.9932**.
2. **Support Vector Machine (SVM RBF):** Se ubica en el segundo lugar con `{'C': 1, 'gamma': 1}`, logrando un **Accuracy de 0.9909** (Error de clasificación de **0.0091**) y un **Macro F1 de 0.9909**, seguido muy de cerca por **Random Forest** (`Accuracy = 0.9886` y `Macro F1 = 0.9887`). Estos resultados confirman que las fronteras no lineales capturan con mayor precisión la interacción entre las propiedades del suelo y el clima, sirviendo como línea base para evaluar posteriormente la selección de características y la reducción de dimensionalidad con PCA y UMAP.